# Stage 2 — GRPO Self-Play in Pure Sim

**Goal:** Train Llama-3.2-1B (post-SFT) on the 3-agent negotiation env using **real GRPO**: Monte-Carlo policy gradient with group-relative advantages, KL penalty against frozen SFT reference, PPO-style clip ratio, and entropy regularization.

**Runtime:** ~3.5 hours on A10G small.
**Prereq:** SFT checkpoint at `Jayyyy234/agentgrid-sft`.

**Algorithm components actually implemented:**
- Group sampling: `G=4` rollouts per seed → group-relative advantage `A = (R - mean) / (std + 1e-8)`
- PPO-clip surrogate (ε = 0.2)
- KL penalty β = 0.04 against frozen reference (SFT, accessed via `disable_adapter` context)
- Entropy bonus α = 0.01
- AdamW lr 5e-6, linear warmup 5%, grad-clip 1.0
- Loss masked to completion tokens only (prompt tokens have weight 0)
- Curriculum scaled across run: easy → medium → full
- Adapter checkpoints every 5 groups for resume safety

Run cells **in order**, do not skip Cell 1 (env-var must be set before any `agentgrid_env` import).

In [ ]:
# Trust-decision audit — single-file append across all episodes
# Must be set BEFORE importing agentgrid_env (constant captured at import time)
import os
TRUST_DECISIONS_PATH = '/tmp/workspace/trust_decisions.jsonl'
os.environ['AGENTGRID_TRUST_DECISIONS_PATH'] = TRUST_DECISIONS_PATH
# Wipe stale data from previous runs in this notebook session
if os.path.exists(TRUST_DECISIONS_PATH):
    os.remove(TRUST_DECISIONS_PATH)
print(f'Trust decisions will append to: {TRUST_DECISIONS_PATH}')

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via HF_TOKEN secret.")
else:
    login()  # fallback: interactive prompt
    print("Logged in interactively.")

In [ ]:
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_missing = [p for p in ("transformers", "peft", "bitsandbytes", "accelerate", "trl", "datasets")
            if importlib.util.find_spec(p) is None]
if _missing:
    print(f"Installing: {_missing}")
    _pip("torch==2.5.1+cu118", "torchvision==0.20.1+cu118",
         "--index-url", "https://download.pytorch.org/whl/cu118")
    _pip("transformers", "peft", "accelerate", "bitsandbytes", "trl",
         "datasets", "plotly", "matplotlib", "pandas")
    print("Done.")
else:
    print("All dependencies present.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/tmp/workspace/AgentGrid_V1")

if not REPO_DIR.exists():
    print("Workspace missing — cloning repo...")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone",
        "https://github.com/jayyyyqwq/GaN_J_AI.git", str(REPO_DIR)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
    print("Repo cloned and installed.")

os.chdir(str(REPO_DIR))
print("Working dir:", os.getcwd())


In [ ]:
import json, os, random, time
from pathlib import Path

# ── Hub repos ───────────────────────────────────────────────────────
SFT_CHECKPOINT = "Jayyyy234/agentgrid-sft"
GRPO_REPO      = "Jayyyy234/agentgrid-grpo"

# ── Env shape ───────────────────────────────────────────────────────
AGENTS         = ["A", "B", "C"]
EPISODE_STEPS  = 50

# ── GRPO budget ─────────────────────────────────────────────────────
# Quick sanity run: N_GROUPS=5  (~20 min on A100,  20 episodes)
# Full training:   N_GROUPS=25  (~3.5 h on A10G, 100 episodes)
N_GROUPS       = int(os.environ.get("N_GROUPS", 5))
G              = 4
N_EPISODES     = N_GROUPS * G
CURRICULUM_SCALE = 500 / N_EPISODES

# ── GRPO hyperparameters ─────────────────────────────────────────────
GRPO_LR        = 5e-7
WARMUP_FRAC    = 0.05
CLIP_EPS       = 0.1
KL_BETA        = 0.1
ENT_ALPHA      = 0.01
GRAD_CLIP      = 1.0
TEMP_START     = 0.9
TEMP_END       = 0.7

# ── Sampling / model ────────────────────────────────────────────────
MAX_NEW_TOKENS = 80
MAX_SEQ_LEN    = 2048
LORA_R         = 16

# ── Logging / checkpoints ───────────────────────────────────────────
OUTPUT_DIR     = "/tmp/workspace/grpo_output"
CKPT_DIR       = "/tmp/workspace/grpo_ckpts"
LOG_PATH       = "/tmp/workspace/grpo_rewards.jsonl"
METRICS_PATH   = "/tmp/workspace/grpo_metrics.jsonl"
SAMPLES_PATH   = "/tmp/workspace/grpo_samples.txt"
CKPT_EVERY     = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
for p in [LOG_PATH, METRICS_PATH, SAMPLES_PATH]:
    if os.path.exists(p):
        os.remove(p)

print(f"GRPO config: {N_GROUPS} groups × G={G} = {N_EPISODES} episodes, "
      f"lr={GRPO_LR}, β={KL_BETA}, α={ENT_ALPHA}, clip={CLIP_EPS}")
print(f"Tip: set env var N_GROUPS=25 before running for the full training run.")


## Step 1: Load policy + reference

The policy is the SFT checkpoint with new trainable LoRA adapters on top.
The reference is the same model with adapters disabled (via `peft.disable_adapter()`) — saves ~1 GB VRAM vs loading SFT twice.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

BASE_MODEL = "meta-llama/Llama-3.2-1B"
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# PEFT API differs across versions; prefer non-reentrant checkpointing when available.
try:
    base = prepare_model_for_kbit_training(
        base,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
except TypeError:
    base = prepare_model_for_kbit_training(
        base,
        use_gradient_checkpointing=True,
    )

if hasattr(base, "gradient_checkpointing_enable"):
    try:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        base.gradient_checkpointing_enable()

base.config.use_cache = False
if hasattr(base, "enable_input_require_grads"):
    base.enable_input_require_grads()

try:
    tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

try:
    model = PeftModel.from_pretrained(base, SFT_CHECKPOINT, is_trainable=True)
    print(f"Loaded SFT adapter from {SFT_CHECKPOINT}")
except Exception as e:
    print(f"SFT checkpoint unavailable ({e}).")
    print("Attaching a fresh LoRA adapter to the base model instead.")
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)

model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

# Verify disable_adapter() works (needed for KL reference)
test_input = tokenizer("test", return_tensors="pt").to(model.device)
with torch.no_grad():
    logits_policy = model(**test_input).logits
    with model.disable_adapter():
        logits_ref = model(**test_input).logits
diff = (logits_policy - logits_ref).abs().max().item()
print(f"Adapter on/off logit-diff: {diff:.4f}  (>0 means adapter is active)")
print("disable_adapter() ✓ working.")

## Step 2: Helpers — observation builder, parser, sampler, log-prob computer

These functions are the contract between the SFT model and the env. The observation JSON schema and the action JSON schema must match `training/generate_sft_data.py` exactly, or the model produces garbage.

In [ ]:
import json, math, random
import numpy as np
import torch
import torch.nn.functional as F
from agentgrid_spaces.runner import HeadlessRunner
from agentgrid_env.server.agentgrid_environment import _spawn_task

ACTION_NAMES = ("broadcast", "make_offer", "accept_offer", "execute_task", "renege", "idle")

ACTION_KWARGS: dict[str, set[str]] = {
    "broadcast":    {"message"},
    "make_offer":   {"to", "give_type", "give_amount", "want_type", "want_amount"},
    "accept_offer": {"offer_id"},
    "execute_task": set(),
    "renege":       {"offer_id"},
    "idle":         set(),
}

# Fields that have no default in the runner — action falls back to idle if missing
_REQUIRED: dict[str, set[str]] = {
    "make_offer": {"to"},
}


def build_obs_json(env, agent_id: str) -> dict:
    peers = [p for p in AGENTS if p != agent_id]
    task = env._tasks[agent_id]
    trust_snap = env._trust[agent_id].snapshot_for_obs()
    inbox = [{"from": m["from"], "message": m["message"]}
             for m in env._message_bus if m["from"] != agent_id]
    pending_offers = [
        {"offer_id": oid, "from": o["from"], "to": o["to"],
         "give_type": o["give_type"], "give_amount": o["give_amount"],
         "want_type": o["want_type"], "want_amount": o["want_amount"]}
        for oid, o in env._offers.items()
        if o["to"] == agent_id and o["status"] == "pending"
    ]
    trust_block = {
        p: {
            "Q_accept": round(float(trust_snap.get(f"Q_accept_{p}", 0.0)), 2),
            "Q_trust":  round(float(trust_snap.get(f"Q_trust_{p}",  0.0)), 2),
            "UCB":      round(float(trust_snap.get(f"UCB_{p}",      9.99)), 2),
            "N":        int(trust_snap.get(f"N_{p}", 0)),
        }
        for p in peers
    }
    return {
        "YOUR_AGENT": agent_id,
        "STEP": env._game_step,
        "BATTERY": round(float(env._batteries[agent_id]), 2),
        "TASK": {"energy_cost": task["energy_cost"], "reward_if_done": task["reward_if_done"]},
        "PEERS": {p: {"battery": round(float(env._batteries[p]), 2),
                      "reputation": round(float(env._reputation[p]), 2)} for p in peers},
        "TRUST_MODEL": trust_block,
        "INBOX": inbox,
        "PENDING_OFFERS": pending_offers,
    }


def pick_hint(obs: dict) -> str:
    has_offer = len(obs["PENDING_OFFERS"]) > 0
    low_battery = obs["BATTERY"] < 0.4
    pool = ["execute_task", "idle", "broadcast"]
    if has_offer:
        pool += ["accept_offer", "accept_offer"]
    if low_battery or random.random() < 0.5:
        pool += ["make_offer"]
    return random.choice(pool)


def build_prompt(obs: dict, hint: str) -> str:
    return (
        f"<s>[INST] {json.dumps(obs, indent=2)}\n\n"
        f"Hint: a reasonable next action right now is `{hint}`.\n"
        f"Reply with the JSON action. [/INST]"
    )


def parse_action(text: str, agent_id: str) -> tuple[str, dict]:
    s = text.strip()
    if s.startswith("```"):
        s = s.split("```", 2)[1]
        if s.startswith("json"):
            s = s[4:]
        s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1 or b <= a:
        return "idle", {}
    try:
        obj = json.loads(s[a:b + 1])
    except json.JSONDecodeError:
        return "idle", {}
    if not isinstance(obj, dict):
        return "idle", {}
    name = obj.get("action")
    if name not in ACTION_NAMES:
        return "idle", {}
    kwargs = {k: v for k, v in obj.items() if k in ACTION_KWARGS[name]}
    # Fall back to idle if a required positional arg is missing
    if not _REQUIRED.get(name, set()).issubset(kwargs):
        return "idle", {}
    return name, kwargs


def tokenize_prompt(prompt: str) -> torch.Tensor:
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids
    return ids.to(model.device)


@torch.no_grad()
def sample_action(prompt_ids: torch.Tensor, temperature: float) -> tuple[str, torch.Tensor]:
    """Sample completion. Returns (text, cpu completion_ids). No logprob stored."""
    out = model.generate(
        prompt_ids,
        attention_mask=torch.ones_like(prompt_ids),
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=max(temperature, 0.01),
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    c_ids = out[:, prompt_ids.shape[1]:]
    return tokenizer.decode(c_ids[0], skip_special_tokens=True), c_ids.cpu()


def compute_logp(prompt_ids: torch.Tensor, completion_ids: torch.Tensor) -> torch.Tensor:
    """Per-token log-prob of completion under the current adapter. Requires grad."""
    full = torch.cat([prompt_ids, completion_ids], dim=1)
    P, T = prompt_ids.shape[1], completion_ids.shape[1]
    logits = model(full, attention_mask=torch.ones_like(full)).logits[0, P-1:P-1+T].float()
    lp = F.log_softmax(logits, dim=-1)
    return lp.gather(-1, completion_ids[0].unsqueeze(-1)).squeeze(-1)  # (T,)


@torch.no_grad()
def compute_ref_logp(prompt_ids: torch.Tensor, completion_ids: torch.Tensor) -> torch.Tensor:
    """Per-token log-prob under the frozen reference (adapter disabled)."""
    with model.disable_adapter():
        return compute_logp(prompt_ids, completion_ids).detach()


print("Helpers defined.")


### Smoke test: 1 episode, dump 5 samples

Confirms the format pipeline works **before** burning A10G time on training. If samples show all `idle` or all parse-fails, fix it here — not 30 minutes into the run.

In [ ]:
model.eval()
runner = HeadlessRunner(episode_steps=EPISODE_STEPS)
snap = runner.reset(seed=0)

action_counts = {n: 0 for n in ACTION_NAMES}
parse_fails = 0
sample_lines = []

step = 0
while not snap.done:
    for agent_id in AGENTS:
        obs = build_obs_json(runner._env, agent_id)
        hint = pick_hint(obs)
        prompt = build_prompt(obs, hint)
        prompt_ids = tokenize_prompt(prompt)
        completion, c_ids = sample_action(prompt_ids, temperature=TEMP_START)
        action, kwargs = parse_action(completion, agent_id)
        action_counts[action] += 1
        if action == "idle" and "idle" not in completion.lower():
            parse_fails += 1
        try:
            runner.apply(agent_id, action, **kwargs)
        except Exception as e:
            runner.apply(agent_id, "idle")
            print(f"  [smoke] {agent_id} {action} {kwargs} → {e}")
        if len(sample_lines) < 3:
            sample_lines.append(f"[{agent_id} s={step}] hint={hint} → {action}  |  {completion[:80]!r}")
    snap = runner.snapshot()
    step += 1

runner._env._dump_trust_decisions()

print("Action distribution:", action_counts)
print(f"Parse fails: {parse_fails}")
print(f"Return: {sum(snap.rewards.values())/3:.2f}  pk: {snap.promise_keep_ratio:.2f}")
for s in sample_lines:
    print(s)

non_idle = sum(v for k, v in action_counts.items() if k != "idle")
if non_idle < 5:
    print(f"\nWARNING: only {non_idle} non-idle actions — model may not be following the format yet. Proceeding anyway.")
else:
    print(f"\n✓ Smoke test passed — {non_idle} non-idle actions.")


## Step 3: GRPO training loop

For each of `N_GROUPS` groups:
1. Roll out `G=4` trajectories with the **same seed** (different sub-seeds for stochasticity), capturing per-token `logp_old` during sampling.
2. Compute group-relative advantages `A_i = (R_i − mean) / (std + ε)`.
3. Re-run forward with grad enabled to get `logp_cur` and `entropy`; query frozen reference via `disable_adapter()` to get `logp_ref`.
4. Loss = PPO-clipped surrogate + β·KL(π‖π_ref) − α·H(π), masked to completion tokens.
5. Backprop, grad-clip, optimizer step.
6. Append per-episode reward log (locked schema) and per-group metrics; checkpoint adapters every 5 groups.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

trainable_params = [p for p in model.parameters() if p.requires_grad]
if not trainable_params:
    raise RuntimeError(
        "No trainable parameters found. Rerun the model-setup cell (Step 1) so LoRA is attached with is_trainable=True."
    )

optimizer = AdamW(trainable_params, lr=GRPO_LR, betas=(0.9, 0.95), weight_decay=0.0)
scheduler = CosineAnnealingLR(optimizer, T_max=N_GROUPS, eta_min=GRPO_LR * 0.1)

runner = HeadlessRunner(episode_steps=EPISODE_STEPS)
t_start = time.time()
ep_global = 0

def temp_for_group(g_idx: int) -> float:
    return TEMP_START + (TEMP_END - TEMP_START) * g_idx / max(1, N_GROUPS - 1)


for group_idx in range(N_GROUPS):
    temperature = temp_for_group(group_idx)

    # ── 1. Rollout (no gradients) ────────────────────────────────────
    model.eval()
    episodes: list[dict] = []

    for g in range(G):
        snap = runner.reset(seed=group_idx * 1000 + g)
        scaled = int(ep_global * CURRICULUM_SCALE)
        runner._env._total_episodes_completed = scaled
        runner._env._tasks = {a: _spawn_task(runner._env._rng, scaled) for a in AGENTS}

        steps: list[tuple[torch.Tensor, torch.Tensor]] = []

        while not snap.done:
            for agent_id in AGENTS:
                obs   = build_obs_json(runner._env, agent_id)
                p_ids = tokenize_prompt(build_prompt(obs, pick_hint(obs)))
                if p_ids.shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
                    p_ids = p_ids[:, -(MAX_SEQ_LEN - MAX_NEW_TOKENS):]
                completion, c_ids = sample_action(p_ids, temperature)
                action, kwargs    = parse_action(completion, agent_id)
                try:
                    runner.apply(agent_id, action, **kwargs)
                except Exception as e:
                    # Malformed kwargs that slipped past parse_action — idle and continue
                    runner.apply(agent_id, "idle")
                    print(f"  [runner] {agent_id} {action} {kwargs} → {e}")
                if c_ids.shape[1] > 0:
                    steps.append((p_ids.cpu(), c_ids))
            snap = runner.snapshot()

        runner._env._dump_trust_decisions()
        ep_return = sum(snap.rewards.values()) / len(AGENTS)
        episodes.append({"steps": steps, "return": ep_return,
                         "rewards": dict(snap.rewards),
                         "pk": float(snap.promise_keep_ratio)})

        with open(LOG_PATH, "a") as f:
            f.write(json.dumps({"rewards": snap.rewards,
                                "promise_keep": snap.promise_keep_ratio}) + "\n")
        ep_global += 1

    # ── 2. Group-relative advantages ────────────────────────────────
    returns    = np.array([e["return"] for e in episodes], dtype=np.float64)
    mean_R     = float(returns.mean())
    std_R      = float(returns.std()) + 1e-8
    advantages = (returns - mean_R) / std_R

    # ── 3. Gradient phase (REINFORCE + clipped KL anchor) ───────────
    model.train()
    optimizer.zero_grad(set_to_none=True)

    sum_pl  = 0.0
    sum_kl  = 0.0
    n_valid = 0
    n_skip  = 0

    for ep, adv in zip(episodes, advantages):
        adv_t = torch.tensor(float(adv), device=model.device, dtype=torch.float32)

        for p_ids_cpu, c_ids_cpu in ep["steps"]:
            p_ids = p_ids_cpu.to(model.device)
            c_ids = c_ids_cpu.to(model.device)

            try:
                logp_cur = compute_logp(p_ids, c_ids)
                logp_ref = compute_ref_logp(p_ids, c_ids)

                T = min(logp_cur.shape[0], logp_ref.shape[0])
                if T == 0:
                    n_skip += 1
                    continue

                policy_loss = -(adv_t * logp_cur[:T]).mean()
                kl          = (logp_ref[:T] - logp_cur[:T]).clamp(-10, 10).mean()
                (policy_loss + KL_BETA * kl).backward()

                sum_pl  += policy_loss.item()
                sum_kl  += kl.item()
                n_valid += 1

            except RuntimeError as e:
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                n_skip += 1
                print(f"  [skip] {e}")
                continue

    if n_valid > 0:
        for p in trainable_params:
            if p.grad is not None:
                p.grad.div_(n_valid)

    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP).item()

    if math.isnan(grad_norm) or math.isinf(grad_norm):
        print(f"  ⚠ [g={group_idx}] NaN/Inf gradient — skipping step")
        optimizer.zero_grad(set_to_none=True)
    else:
        optimizer.step()

    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    # ── 4. Checkpoint every group ────────────────────────────────────
    ckpt_path = os.path.join(CKPT_DIR, f"group_{group_idx:03d}")
    model.save_pretrained(ckpt_path)

    # ── 5. Logging ───────────────────────────────────────────────────
    avg_pl  = sum_pl / max(1, n_valid)
    avg_kl  = sum_kl / max(1, n_valid)
    cur_lr  = scheduler.get_last_lr()[0]
    elapsed = (time.time() - t_start) / 60
    pk_mean = sum(e["pk"] for e in episodes) / len(episodes)

    with open(METRICS_PATH, "a") as f:
        f.write(json.dumps({
            "group": group_idx, "ep_global": ep_global,
            "policy_loss": avg_pl, "kl": avg_kl,
            "grad_norm": grad_norm, "lr": cur_lr,
            "group_mean_return": mean_R, "group_std_return": std_R,
            "temperature": temperature, "elapsed_min": elapsed,
            "n_valid": n_valid, "n_skip": n_skip,
        }) + "\n")

    print(f"[g={group_idx:3d}/{N_GROUPS}] "
          f"R={mean_R:+.2f}±{std_R:.2f}  pk={pk_mean:.2f}  "
          f"PL={avg_pl:+.4f}  KL={avg_kl:.4f}  "
          f"gn={grad_norm:.3f}  lr={cur_lr:.2e}  "
          f"t={elapsed:.1f}m  ok={n_valid}  skip={n_skip}  "
          f"ckpt→{ckpt_path}")

    if avg_kl > 2.0:
        print(f"  ⚠ KL={avg_kl:.2f} very high — consider raising KL_BETA or lowering lr")
    if math.isnan(avg_pl):
        print("  ⚠ NaN loss — stopping.")
        break

    del episodes
    torch.cuda.empty_cache()

print(f"\nDone. {ep_global} episodes, {N_GROUPS} groups, "
      f"{(time.time()-t_start)/60:.1f} min total.")
print(f"Latest checkpoint: {os.path.join(CKPT_DIR, f'group_{N_GROUPS-1:03d}')}" )

## Step 4: Plots — reward curve + training health

Two outputs:
- **`grpo_curve.png`** — episode return + smoothed mean (the "did training work" plot)
- **`grpo_training_health.png`** — KL, entropy, clip fraction, group-mean return (the "is training stable" diagnostic)

Both consume the per-group metrics file `grpo_metrics.jsonl` and per-episode `grpo_rewards.jsonl`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load per-episode rewards
ep_rewards = []
ep_promise = []
with open(LOG_PATH) as f:
    for line in f:
        d = json.loads(line)
        ep_rewards.append(sum(d["rewards"].values()) / 3)
        ep_promise.append(d["promise_keep"])
ep_rewards = np.array(ep_rewards)
ep_promise = np.array(ep_promise)

# Load per-group metrics
metrics = []
with open(METRICS_PATH) as f:
    for line in f:
        metrics.append(json.loads(line))

# ── Plot 1: reward curve ────────────────────────────────────────────
window = max(5, len(ep_rewards) // 20)
smooth = np.convolve(ep_rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(ep_rewards, alpha=0.3, label="Episode return")
plt.plot(range(window - 1, len(ep_rewards)), smooth, label=f"Smoothed (w={window})")
plt.xlabel("Episode")
plt.ylabel("Avg episode return (3 agents)")
plt.title("GRPO Self-Play — AgentGrid Sim")
plt.legend()
plt.tight_layout()
plt.savefig("/tmp/workspace/grpo_curve.png", dpi=150)
plt.show()
print("Saved: /tmp/workspace/grpo_curve.png")

# ── Plot 2: training health (2x2) ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
groups = [m["group"] for m in metrics]

axes[0, 0].plot(groups, [m["group_mean_return"] for m in metrics], "o-", label="group mean")
axes[0, 0].fill_between(
    groups,
    [m["group_mean_return"] - m["group_std_return"] for m in metrics],
    [m["group_mean_return"] + m["group_std_return"] for m in metrics],
    alpha=0.2,
)
axes[0, 0].set_title("Group-mean return ± std")
axes[0, 0].set_xlabel("Group"); axes[0, 0].set_ylabel("Return")

axes[0, 1].plot(groups, [m["kl"] for m in metrics], "o-")
axes[0, 1].axhline(KL_BETA * 10, color="red", ls=":", alpha=0.5, label="alarm threshold")
axes[0, 1].set_title("KL(π‖π_ref) — should plateau")
axes[0, 1].set_xlabel("Group"); axes[0, 1].set_ylabel("KL")
axes[0, 1].legend()

axes[1, 0].plot(groups, [m["grad_norm"] for m in metrics], "o-")
axes[1, 0].axhline(GRAD_CLIP, color="red", ls=":", alpha=0.5, label=f"clip={GRAD_CLIP}")
axes[1, 0].set_title("Gradient norm — should stay ≤ clip threshold")
axes[1, 0].set_xlabel("Group"); axes[1, 0].set_ylabel("Grad norm")
axes[1, 0].legend()

skip_rate = [m["n_skip"] / max(1, m["n_valid"] + m["n_skip"]) for m in metrics]
axes[1, 1].plot(groups, [m["policy_loss"] for m in metrics], "o-", label="policy loss")
axes[1, 1].plot(groups, skip_rate, "s-", label="skip rate")
axes[1, 1].set_title("Policy loss & skip rate (lower skip is better)")
axes[1, 1].set_xlabel("Group")
axes[1, 1].legend()

plt.suptitle(f"GRPO Training Health — {N_GROUPS} groups × G={G}")
plt.tight_layout()
plt.savefig("/tmp/workspace/grpo_training_health.png", dpi=150)
plt.show()
print("Saved: /tmp/workspace/grpo_training_health.png")

# ── Pass/fail summary ─────────────────────────────────────────────────
last_n = min(100, len(ep_rewards))
last_n_pk = min(100, len(ep_promise))
mean_R  = float(ep_rewards[-last_n:].mean())
mean_PK = float(ep_promise[-last_n_pk:].mean())
print(f"\n── Submission pass/fail ──")
print(f"Last-{last_n} mean return    : {mean_R:.2f}  (random baseline ~ −2.1)")
print(f"Last-{last_n_pk} mean promise-keep: {mean_PK:.2f}  (target > 0.40) — "
      f"{'PASS ✓' if mean_PK > 0.40 else 'FAIL ✗'}")


## Step 5: Sanity-check artifacts and push checkpoint

Before pushing to Hub, confirm artifact schemas match what the eval scripts expect.

In [ ]:
# ── Artifact sanity checks ────────────────────────────────────────
import os, json

print("Artifacts produced:")
for path in [LOG_PATH, METRICS_PATH, TRUST_DECISIONS_PATH,
             "/tmp/workspace/grpo_curve.png", "/tmp/workspace/grpo_training_health.png"]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {'✓' if exists else '✗'} {path}  ({size:,} bytes)")

# Verify grpo_rewards.jsonl schema (consumed by eval/plot_three_curves.py)
with open(LOG_PATH) as f:
    first_line = f.readline()
d = json.loads(first_line)
assert set(d) == {"rewards", "promise_keep"}, f"Wrong keys: {set(d)}"
assert len(d["rewards"]) == 3, f"Expected 3-agent rewards, got {len(d['rewards'])}"
n_eps = sum(1 for _ in open(LOG_PATH))
print(f"\n✓ grpo_rewards.jsonl: {n_eps} episodes, schema OK")

# Verify trust_decisions.jsonl schema (consumed by eval/plot_trust_correlation.py)
if os.path.exists(TRUST_DECISIONS_PATH):
    with open(TRUST_DECISIONS_PATH) as f:
        first = f.readline()
    if first.strip():
        td = json.loads(first)
        required = {"Q_chosen", "Q_alternatives", "step"}
        assert required <= set(td), f"Missing keys: {required - set(td)}"
        n_td = sum(1 for _ in open(TRUST_DECISIONS_PATH))
        print(f"✓ trust_decisions.jsonl: {n_td} events, schema OK")
    else:
        print("⚠ trust_decisions.jsonl is empty — no accept_offer fired during training")
else:
    print("⚠ trust_decisions.jsonl missing")

print("\nDownload these from Colab to local repo (place under eval/plots/):")
print(f"  - {LOG_PATH}              → eval/plots/grpo_rewards.jsonl")
print(f"  - {TRUST_DECISIONS_PATH} → eval/plots/trust_decisions.jsonl")
print(f"  - /tmp/workspace/grpo_curve.png            → eval/plots/grpo_curve.png")
print(f"  - /tmp/workspace/grpo_training_health.png  → eval/plots/grpo_training_health.png")

In [ ]:
# Push GRPO checkpoint to HF Hub
# Use a fresh write-scoped token from https://huggingface.co/settings/tokens
# DO NOT paste the token in the notebook — pass via prompt or env var.
from huggingface_hub import login

# Option A (interactive): prompts for token
login()
# Option B (env var): set HF_TOKEN before launching, then:
# login(token=os.environ["HF_TOKEN"])

model.push_to_hub(GRPO_REPO)
tokenizer.push_to_hub(GRPO_REPO)
print(f"GRPO checkpoint saved to hf.co/{GRPO_REPO}")